# Agentic AI and RAGs ?

Using only open source frameworks.
- Tiny KB with FAISS + FakeEmbeddings
- Free Wikipedia utility (no API keys)
- Rule-based planner
- Stub summarizer with optional tiny HF model (sshleifer/tiny-gpt2)
Run all cells top to bottom in Colab.

In [ ]:
!pip install -U -q langchain langchain-community faiss-cpu wikipedia transformers accelerate sentencepiece

## 1) Build the KB retriever

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FakeEmbeddings

kb_docs = [
    Document(
        page_content="""
Agentic systems reason step-by-step about which tools to call instead of invoking tools blindly.
Their core loop is: (1) interpret the user goal, (2) inspect available context, (3) decide whether tools are needed,
(4) call one or more tools in a planned sequence, and (5) synthesize an answer grounded in the tool results.
""".strip(),
        metadata={"source": "kb:agentic_concept"},
    ),
    Document(
        page_content="""
Retrievers fetch grounding passages from a knowledge base and are the primary interface to internal documents.
Given a user query, the system should: (1) normalize and possibly expand the query, (2) retrieve top-k candidates,
(3) read them carefully, and (4) base the answer primarily on those passages.
""".strip(),
        metadata={"source": "kb:retrievers"},
    ),
    Document(
        page_content="""
Wikipedia is a broad-coverage, free fallback when the curated knowledge base lacks coverage or appears incomplete.
""".strip(),
        metadata={"source": "kb:wikipedia_tip"},
    ),
    Document(
        page_content="""
When evidence is thin, ambiguous, or conflicting, the system must be transparent about uncertainty.
""".strip(),
        metadata={"source": "kb:honesty"},
    ),
    Document(
        page_content="""
Answers should be concise, focused, and well-structured, typically within 2–4 sentences.
""".strip(),
        metadata={"source": "kb:style"},
    ),
]

embeddings = FakeEmbeddings(size=256)
vs = FAISS.from_documents(kb_docs, embeddings)
retriever = vs.as_retriever(search_kwargs={"k": 3})
print("KB ready with", len(kb_docs), "docs")

KB ready with 5 docs


## 2) Open source external tool: Wikipedia search

In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper

wiki = WikipediaAPIWrapper(lang='en', top_k_results=2, doc_content_chars_max=1000)

def wiki_search(query: str, k: int = 2):
    try:
        results = wiki.load(query)
        snippets = [{'title': r.metadata['title'], 'summary': r.page_content} for r in results[:k]]
        return snippets, None
    except Exception as e:
        return [], str(e)

print(wiki_search('Python programming')[0][:1])

[{'title': 'Python (programming language)', 'summary': 'Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation. Python is dynamically type-checked and garbage-collected. It supports multiple programming paradigms, including structured (particularly procedural), object-oriented and functional programming.\nGuido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revi'}]


## 3) Simple planner (rule-based)

In [ ]:
kb_keywords = ['agentic', 'retriever', 'citation', 'ground', 'transparen', 'honest', 'style']

def plan(question: str):
    q_lower = question.lower()
    if any(k in q_lower for k in kb_keywords):
        return {'action': 'kb'}
    return {'action': 'wiki'}

print(plan('How to ground answers?'))
print(plan('Who created Python?'))

{'action': 'kb'}
{'action': 'wiki'}


## 4) Answer function with stub or tiny HF model

In [5]:
try:
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_community.chat_models.fake import FakeListChatModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
    import torch
except ModuleNotFoundError as e:
    print(f"[ERROR] Missing module: {e.name}")
    print("\n--- ACTION REQUIRED ---")
    print("1. Ensure the first cell (!pip install...) finished successfully.")
    print("2. Go to 'Runtime' > 'Restart session'.")
    print("3. Run all cells again to refresh the environment.")
    raise

prompt = ChatPromptTemplate.from_template("You are a helpful agentic assistant. Use the given context and wiki snippets.\n"
    "If there is little evidence, say so and suggest a follow-up query.\n"
    "Cite sources like [kb:doc1] or [wiki:Title].\n"
    "Question: {question}\n"
    "Context:{context}\n"
    "Wiki:{wiki}\n"
    "Answer:"
)

def get_tiny_generator(model_id: str = 'sshleifer/tiny-gpt2'):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    return pipeline('text-generation', model=model, tokenizer=tok, device=-1)

def summarize_with_tiny(prompt_text: str, max_new_tokens: int = 50):
    gen = get_tiny_generator()
    out = gen(prompt_text, max_new_tokens=max_new_tokens, truncation=True, pad_token_id=50256)
    full = out[0]['generated_text']
    completion = full[len(prompt_text):].strip()
    return completion

def answer_question(question: str, use_tiny_model: bool = False):
    pl = plan(question)
    docs = retriever.invoke(question) if pl['action'] == 'kb' else []
    wiki_snips = []
    wiki_err = None
    if pl['action'] == 'wiki':
        wiki_snips, wiki_err = wiki_search(question)

    context_text = '\n'.join([f"[{d.metadata.get('source')}] {d.page_content}" for d in docs]) or 'No KB context.'
    wiki_text = '\n'.join([f"[wiki:{s['title']}] {s['summary']}" for s in wiki_snips]) or 'No wiki snippets.'

    messages = prompt.format_messages(question=question, context=context_text, wiki=wiki_text)
    prompt_val = messages[0].content

    if use_tiny_model:
        final_answer = summarize_with_tiny(prompt_val)
    else:
        stub = FakeListChatModel(responses=['Based on [kb:retrievers], grounding requires fetching top-k candidates and reading them carefully.'])
        final_answer = stub.invoke(prompt_val).content

    return {
        'plan': pl,
        'kb_sources': [d.metadata.get('source') for d in docs],
        'wiki_sources': [s.get('title') for s in wiki_snips],
        'wiki_error': wiki_err,
        'answer': final_answer,
    }

[ERROR] Missing module: langchain_community

--- ACTION REQUIRED ---
1. Ensure the first cell (!pip install...) finished successfully.
2. Go to 'Runtime' > 'Restart session'.
3. Run all cells again to refresh the environment.


ModuleNotFoundError: No module named 'langchain_community'

## 5) Quick check on sample questions

In [3]:
# The NameError happens because cell 7956fc57 failed to define the function due to a missing library.
# Please ensure you have restarted the runtime after running the !pip install cell.

tests = [
    "What are the key principles of agentic systems?",
    "Who was the founder of the Python programming language?",
    "How many people live in Mars?"
]

try:
    for q in tests:
        res = answer_question(q, use_tiny_model=False)
        print(f"\n--- Query: {q} ---")
        print(f"Plan Action: {res['plan']['action']}")
        print(f"Sources: {res['kb_sources'] + res['wiki_sources']}")
        print(f"Answer: {res['answer']}")
except NameError:
    print("DIAGNOSIS: 'answer_question' is not defined.")
    print("REASON: Cell 7956fc57 failed to run because 'langchain_community' was not found.")
    print("FIX: Restart your Colab runtime (Runtime > Restart session) and run the !pip install cell again.")

DIAGNOSIS: 'answer_question' is not defined.
REASON: Cell 7956fc57 failed to run because 'langchain_community' was not found.
FIX: Restart your Colab runtime (Runtime > Restart session) and run the !pip install cell again.
